# Qwen3.5-9B FineTuning — NL to ASP Translation

In [ ]:
from huggingface_hub import login
login('YOUR HUGGINGFACE_TOKEN')

In [ ]:
import os
import json
from pathlib import Path
from datasets import load_dataset
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from transformers import EarlyStoppingCallback
import trl
from trl import SFTTrainer, SFTConfig
import torch
import peft
from peft import LoraConfig


print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("Torch:", torch.__version__)


## Dataset Generation


In [ ]:
def load_data(path, test_size=0.1, seed=42):
    """
    Load JSON dataset and convert to conversational format.
    Expects file structure: {"data_dict": [{"NL_V2": ..., "CNL_V2": ...}, ...]}
    Returns (train_dataset, test_dataset). If test_size=0, returns (full_dataset, None).
    """
    def create_conversation(sample):
        # FIX 1b: sample is already a flat record after field="data_dict"
        # Use sample['NL_V2'] directly — NOT sample['data_dict']['NL_V2']
        return {
            "messages": [
                {
                    "role": "system",
                    "content": "You are an expert in Translating the Natural language (NL) into Answer Set Programming (ASP) translation. Always provide precise, syntactically and semantically correct translations of NL into ASP."
                },
                {
                    "role": "user",
                    "content": f"Translate the following natural language to Answer Set Programming: {sample['NL_V2']}"
                },
                {
                    "role": "assistant",
                    "content": sample["ASP"]
                },
            ]
        }

    dataset = load_dataset("json", data_files=path, field="data_dict", split="train")
    dataset = dataset.map(create_conversation, remove_columns=dataset.features, batched=False)
    print("Dataset converted to conversational format.")
    print(f"Total samples: {len(dataset)}")

    if test_size == 0:
        return dataset, None

    split_dataset = dataset.train_test_split(test_size=test_size, seed=seed)
    train_dataset = split_dataset["train"].shuffle(seed=seed)
    test_dataset  = split_dataset["test"]

    return train_dataset, test_dataset

In [ ]:
dataset_file = "Path/To/Your/train_data.json" 
train_data, test_data = load_data(dataset_file, test_size=0.1)

# Optional: inspect an example
print("Example from train set:")
print(train_data[2])

print(f"Train dataset size: {len(train_data)}")
print(f"Test dataset size:  {len(test_data)}")

## Model Loading

In [ ]:


base_model_name = "Qwen/Qwen3.5-9B"

def load_prepare_model(model_name=base_model_name):
    """
    Load Qwen3.5-9B in bfloat16 on ROCm GPU.
    Uses dtype= (not deprecated torch_dtype=).
    flash-linear-attention not available on ROCm — torch fallback is used (expected).
    """
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.bfloat16,      # correct param for transformers 5.x
        device_map="auto",
        trust_remote_code=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    print(f"Model loaded: {model_name}")
    print(f"GPU memory:   {torch.cuda.memory_allocated() / 1e9:.2f} GB")

    return model, tokenizer

In [ ]:
base_model = base_model_name
model, tokenizer = load_prepare_model(model_name=base_model)

## Preprocessing — Completion-Only Format

In [ ]:
def preprocess_to_completion_format(dataset, tokenizer):
    """
    Convert messages to prompt+completion format for SFTConfig(completion_only_loss=True).
    Loss computed ONLY on the CNL completion tokens — system/user turns are masked.
    """
    # Qwen3.5 uses the same ChatML format as Qwen3
    ASSISTANT_HEADER = "<|im_start|>assistant\n"

    def convert(sample):
        full_text = tokenizer.apply_chat_template(
            sample["messages"],
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False,   # suppress <think> block during training
        )

        split_idx = full_text.rfind(ASSISTANT_HEADER)
        if split_idx == -1:
            raise ValueError(
                f"Assistant header not found in template output.\nText: {full_text[:200]}"
            )

        prompt     = full_text[:split_idx + len(ASSISTANT_HEADER)]
        completion = full_text[split_idx + len(ASSISTANT_HEADER):]

        return {"prompt": prompt, "completion": completion}

    return dataset.map(convert, remove_columns=dataset.column_names)


train_data_processed = preprocess_to_completion_format(train_data, tokenizer)
test_data_processed  = preprocess_to_completion_format(test_data,  tokenizer)

# Verify split is correct
sample = train_data_processed[0]
print("--- PROMPT ---")
print(sample["prompt"])
print("--- COMPLETION ---")
print(sample["completion"])

## Trainer Setup & Training

In [ ]:
max_seq_length = 1024

def create_trainer(model, tokenizer, train_dataset, eval_dataset=None,
                   max_seq_length=max_seq_length, num_epochs=4, early_stopping_patience=2):
    """
    SFTTrainer configured for Qwen3.5-9B on ROCm.
    - completion_only_loss=True: loss on CNL output tokens only
    """
    peft_config = LoraConfig(
        r=16,  # 16, 8, 32
        lora_alpha=32,  #32,16
        lora_dropout=0.1, #0.1, 0.15
        use_rslora=True,
        use_dora=False,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ],
        task_type="CAUSAL_LM",
    )

    training_arguments = SFTConfig(
        output_dir="PATH/TO/SAVE/ADAPTER",
        num_train_epochs=num_epochs,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=1,
        optim="adamw_torch",
        logging_steps=25,
        learning_rate=1e-4, #1e-4 , 5e-5
        weight_decay=0.02, #0.02 , 0.05, 
        fp16=False,
        bf16=True,                  # ROCm supports bfloat16
        max_grad_norm=0.3,
        max_steps=-1,
        warmup_ratio=0.03, 
        lr_scheduler_type="cosine",
        neftune_noise_alpha=5.0, 
        load_best_model_at_end=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        max_length=max_seq_length,
        completion_only_loss=True,  # mask system+user tokens from loss
        dataset_text_field=None,    # required when using prompt+completion format
        report_to="none",
    )

    callbacks = []
    if eval_dataset is not None and early_stopping_patience > 0:
        callbacks.append(EarlyStoppingCallback(early_stopping_patience=early_stopping_patience))

    trainer = SFTTrainer(
        model=model,
        args=training_arguments,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        peft_config=peft_config,
        processing_class=tokenizer,
        callbacks=callbacks if callbacks else None,
    )

    return trainer

In [ ]:
trainer = create_trainer(
    model, tokenizer,
    train_dataset=train_data_processed,
    eval_dataset=test_data_processed,
    num_epochs=15,
    early_stopping_patience=3
)

trainer.train()

print("Best checkpoint:", trainer.state.best_model_checkpoint)

# Save final adapter
trainer.save_model("PATH/TO/SAVE/ADAPTER")
tokenizer.save_pretrained("PATH/TO/SAVE/ADAPTER")
print("Adapter saved to PATH/TO/SAVE/ADAPTER ")

In [ ]:
trainer.state.best_model_checkpoint  ## path to best checkpoint SAVED